# Redundancy reduction (Prompt #9, Steps 9-12, 47-50)

Average-linkage clusters on d = 1 - |Spearman| (development sample), a representative per cluster by the documented lexicographic rule (null screen, yearly consistency, |median yearly IC|, recent relevance, missingness, cost, warm-up), parameter families reduced to one member or a small diverse subset, and the collinearity of the final sets.

**This notebook contains no calculations.** Everything is computed by `xauusd_quant.selection` and `xauusd_quant.research.feature_selection_*` and read back from `results/feature_selection/`. Generate those first (needs the Prompt #8 research of the same timeframe for the pipeline-null veto):

```
xq feature-select --timeframe 5m                        # every stage + report + manifests
xq feature-select --timeframe 5m --target residual_reduction
xq feature-select --timeframe 5m --target future_return --horizon 5
xq feature-selection-report --timeframe 5m              # summary, SEL-H ledger rows, plots
xq build-feature-manifest --set standard --timeframe 5m # verify + reproduce the matrix
```

**Periods.** Development 2003-2017 selects; validation 2018-2021 only evaluates what development chose and reads the plateaus; the reserved test period (2022-) is never loaded with outcomes. Diagnostic models are linear (ridge, Lasso, elastic net, L1 logistic) on training-CDF rank designs - not the ML system of Prompt #10, and nothing here is a trading rule.

In [ ]:
import sys
import json
from pathlib import Path

# Work against the repository's own source tree, not an installed copy.
ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import polars as pl
from IPython.display import Image, display

from xauusd_quant.selection.config import load_selection_config

scfg = load_selection_config(ROOT / "config" / "feature_selection.yaml")

TIMEFRAME = "5m"       # 5m | 15m (configured); 30m | 1h if they were run

out = scfg.results_path / TIMEFRAME
if not out.exists():
    print(f"No selection for {TIMEFRAME}. Run:  xq feature-select --timeframe {TIMEFRAME}")

def table(name):
    """Load a previously generated selection table (None if it was not written)."""
    path = out / f"{name}.parquet"
    return pl.read_parquet(path) if path.exists() else None

def plot(name):
    path = out / "plots" / f"{name}.png"
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f"(no plot {path.name})")

def load_json(name):
    path = out / name
    return json.loads(path.read_text()) if path.exists() else {}

summary = load_json("summary.json")
sets = load_json("sets.json")
{k: v for k, v in summary.items() if not isinstance(v, (dict, list))}

## Dendrogram (Standard set in bold blue)

In [ ]:
plot('dendrogram')

## The largest clusters and their representatives

In [ ]:
clusters = load_json('redundancy_clusters.json')
pl.DataFrame([{k: c[k] for k in ('cluster', 'size', 'representative', 'mean_abs_spearman', 'why')}
              for c in clusters.get('clusters', []) if c['size'] > 1])

## Parameter families

In [ ]:
table('parameter_families')

## Hard pairs (|rho| >= 0.95) and non-monotone near-duplicates

In [ ]:
pl.DataFrame(clusters.get('hard_pairs', [])).head(20)

In [ ]:
clusters.get('nonmonotone_pairs', [])[:15]

## Effective rank vs feature count

In [ ]:
plot('effective_rank')
table('effective_rank')

## Collinearity of the candidate sets

In [ ]:
table('collinearity')

## Correlation inside the Standard set

In [ ]:
plot('standard_set_correlation')